# Proyecto Parcial 1 - MLY0100
**Tema:** Análisis y Preprocesamiento de Datos de E-commerce Brasileño  
**Integrantes:** Antonio Sepúlveda  
**Fecha:** 13/07/2026

---

## Metodología CRISP-DM

Este proyecto sigue la metodología **CRISP-DM** (Cross-Industry Standard Process for Data Mining) como estándar para abordar el análisis de datos. Las fases que se cubrirán en este notebook son:

1. **Business Understanding** - Entendimiento del negocio
2. **Data Understanding** - Entendimiento de los datos
3. **Data Preparation** - Preparación de los datos

*(Las fases 4-6: Modeling, Evaluation y Deployment se cubrirán en entregas posteriores)*

## Importación de Librerías

Se utilizan las siguientes librerías de Python para Machine Learning y análisis de datos:
- **pandas**: Manipulación y análisis de datos
- **numpy**: Operaciones numéricas y cálculos matemáticos
- **matplotlib**: Visualización básica de datos
- **seaborn**: Visualización estadística avanzada
- **scikit-learn**: Preprocesamiento, encoding y escalado de datos

In [ ]:
# Librerías estándar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Librerías de preprocesamiento
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

# Importar funciones auxiliares del proyecto
import sys
sys.path.append('../src')
from preprocessing import (
    load_csv, save_csv, missing_summary, 
    detect_outliers_iqr, cap_outliers,
    impute_numeric_median, impute_categorical_mode
)
from visualization import hist_plot, box_plot, corr_heatmap
from features import create_ecommerce_features
from data_loader import (
    load_brazilian_ecommerce_datasets, 
    combine_ecommerce_datasets,
    get_dataset_info,
    list_available_datasets
)

print("✓ Librerías importadas correctamente")

---

# FASE 1: BUSINESS UNDERSTANDING
## Entendimiento del Negocio

### Contexto del Negocio

El e-commerce brasileño ha experimentado un crecimiento significativo en los últimos años. Olist es una plataforma de marketplace que conecta pequeñas empresas con clientes en todo Brasil. 

**Objetivos del negocio:**
1. Comprender el comportamiento de compra de los clientes
2. Identificar patrones en los pedidos y productos
3. Mejorar la experiencia del cliente y optimizar operaciones
4. Predecir valores de pedidos para planificación de inventario
5. Clasificar clientes para estrategias de marketing personalizadas

### Preguntas de Negocio

- ¿Cuál es el valor promedio de los pedidos?
- ¿Qué factores influyen en el valor de un pedido?
- ¿Cómo podemos segmentar a los clientes según su comportamiento?
- ¿Qué productos son más populares?
- ¿Existen patrones temporales en las compras?

### Definición de Targets

#### Target para Regresión (Criterio 2 de la Rúbrica)
**Variable objetivo:** `order_total_value` o `price` (valor total del pedido)

**Justificación:**
- Variable numérica continua que permite predecir el valor de compra
- Contexto de negocio: Predecir el valor de pedidos ayuda a:
  - Planificar inventario y logística
  - Optimizar estrategias de pricing
  - Estimar ingresos futuros
  - Identificar pedidos de alto valor para atención especial

#### Target para Clasificación (Criterio 3 de la Rúbrica)
**Variable objetivo:** `order_status_category` o `customer_segment`

**Justificación:**
- Variable categórica que permite clasificar pedidos o clientes
- Contexto de negocio: Clasificar clientes ayuda a:
  - Desarrollar estrategias de marketing segmentadas
  - Mejorar la retención de clientes
  - Personalizar ofertas y promociones
  - Identificar clientes de alto valor

**Nota:** Los targets específicos se ajustarán una vez que se carguen y exploren los datos reales del dataset.

---

# FASE 2: DATA UNDERSTANDING
## Entendimiento de los Datos

En esta fase se realiza un análisis exploratorio completo de los datos para comprender su estructura, calidad y características.

## 2.1 Carga de Datos

Se cargan los datasets del e-commerce brasileño. El dataset típicamente incluye múltiples archivos CSV que deben ser combinados.

In [ ]:
# Cargar datasets principales automáticamente
# Esta función detecta y carga todos los archivos CSV disponibles en data/01_raw/
# NOTA: Los archivos están en mly0100parcial-kedro/data/01_raw/

print("=" * 60)
print("CARGA AUTOMÁTICA DE DATASETS")
print("=" * 60)

# Listar archivos disponibles
# Ruta corregida: apunta a mly0100parcial-kedro/data/01_raw/
available_files = list_available_datasets('../mly0100parcial-kedro/data/01_raw')
print(f"\n📂 Archivos CSV encontrados: {len(available_files)}")
if available_files:
    for f in available_files:
        print(f"  - {f}")
else:
    print("  ⚠ No se encontraron archivos CSV. Verifica que los archivos estén en mly0100parcial-kedro/data/01_raw/")

# Cargar todos los datasets disponibles
datasets = load_brazilian_ecommerce_datasets('../mly0100parcial-kedro/data/01_raw', verbose=True)

# Mostrar resumen de datasets cargados
if datasets:
    print("\n" + "=" * 60)
    print("RESUMEN DE DATASETS CARGADOS")
    print("=" * 60)
    info_df = get_dataset_info(datasets)
    print(info_df.to_string(index=False))
    
    # Guardar datasets individuales en variables para fácil acceso
    if 'orders' in datasets:
        orders_df = datasets['orders']
    if 'order_items' in datasets:
        order_items_df = datasets['order_items']
    if 'customers' in datasets:
        customers_df = datasets['customers']
    if 'products' in datasets:
        products_df = datasets['products']
    if 'sellers' in datasets:
        sellers_df = datasets['sellers']
    if 'payments' in datasets:
        payments_df = datasets['payments']
    if 'reviews' in datasets:
        reviews_df = datasets['reviews']
else:
    print("\n⚠ No se pudieron cargar los datasets. Verifica:")
    print("  1. Que los archivos CSV estén en data/01_raw/")
    print("  2. Que los archivos no estén corruptos")
    print("  3. Que tengas permisos de lectura")

## 2.2 Combinación de Datasets

Se combinan los datasets relacionados para crear un dataset unificado que contenga toda la información necesaria para el análisis.

In [ ]:
# Combinar datasets en un dataset unificado
# Esta función combina automáticamente los datasets relacionados

print("=" * 60)
print("COMBINACIÓN DE DATASETS")
print("=" * 60)

if 'datasets' in locals() and datasets and 'orders' in datasets:
    try:
        # Combinar datasets usando la función auxiliar
        df = combine_ecommerce_datasets(datasets)
        
        print("\n✓ Datasets combinados exitosamente")
        print(f"\n📊 Dimensiones del dataset combinado:")
        print(f"  - Filas: {df.shape[0]:,}")
        print(f"  - Columnas: {df.shape[1]}")
        
        print(f"\n📋 Primeras columnas del dataset combinado:")
        print(f"  {', '.join(df.columns[:10].tolist())}{'...' if len(df.columns) > 10 else ''}")
        
        print(f"\n👀 Primeras filas:")
        display(df.head())
        
        # Mostrar información sobre las combinaciones realizadas
        print(f"\n🔗 Combinaciones realizadas:")
        if 'order_items' in datasets:
            print("  ✓ Orders + Order Items")
        if 'customers' in datasets:
            print("  ✓ + Customers")
        if 'products' in datasets:
            print("  ✓ + Products")
        if 'sellers' in datasets:
            print("  ✓ + Sellers")
        if 'payments' in datasets:
            print("  ✓ + Payments (agregado)")
        
    except Exception as e:
        print(f"\n⚠ Error al combinar datasets: {e}")
        print("Intentando combinación manual...")
        
        # Combinación manual como fallback
        if 'orders' in datasets:
            df = datasets['orders'].copy()
            if 'order_items' in datasets:
                df = df.merge(datasets['order_items'], on='order_id', how='left')
            print("✓ Combinación manual completada")
        else:
            df = pd.DataFrame()
            print("⚠ No se pudo crear el dataset combinado")
else:
    print("⚠ No hay datasets disponibles para combinar")
    print("  Asegúrate de haber ejecutado la celda anterior de carga de datos")
    df = pd.DataFrame()

## 2.3 Análisis de Estructura de Datos

Se analiza la estructura básica del dataset: dimensiones, tipos de datos e información general.

In [ ]:
# Análisis de estructura
if 'df' in locals() and not df.empty:
    print("=" * 60)
    print("INFORMACIÓN GENERAL DEL DATASET")
    print("=" * 60)
    
    print(f"\n📊 Dimensiones:")
    print(f"  - Filas: {df.shape[0]:,}")
    print(f"  - Columnas: {df.shape[1]}")
    
    print(f"\n📋 Información de tipos de datos:")
    df.info()
    
    print(f"\n👀 Primeras filas:")
    df.head()
    
    print(f"\n📝 Nombres de columnas:")
    print(df.columns.tolist())
else:
    print("⚠ Dataset no disponible. Cargar datos primero.")

## 2.4 Estadísticos Descriptivos

### Criterio 8 de la Rúbrica: Estadísticos de Tendencia Central y Dispersión

Se calculan y analizan estadísticos descriptivos para explicar los datos:
- **Tendencia central:** Media, mediana, moda
- **Dispersión:** Desviación estándar, varianza, rango intercuartílico (IQR), rango

In [ ]:
# Estadísticos descriptivos de variables numéricas
if 'df' in locals() and not df.empty:
    print("=" * 60)
    print("ESTADÍSTICOS DESCRIPTIVOS")
    print("=" * 60)
    
    # Seleccionar solo columnas numéricas
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        print(f"\n📊 Resumen estadístico (variables numéricas):")
        desc_stats = df[numeric_cols].describe()
        print(desc_stats)
        
        print(f"\n📈 Estadísticos adicionales:")
        additional_stats = pd.DataFrame({
            'Media': df[numeric_cols].mean(),
            'Mediana': df[numeric_cols].median(),
            'Moda': [df[col].mode()[0] if not df[col].mode().empty else np.nan for col in numeric_cols],
            'Desv. Estándar': df[numeric_cols].std(),
            'Varianza': df[numeric_cols].var(),
            'IQR': df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
            'Rango': df[numeric_cols].max() - df[numeric_cols].min(),
            'Coef. Variación': (df[numeric_cols].std() / df[numeric_cols].mean()) * 100
        })
        print(additional_stats.round(2))
    else:
        print("⚠ No se encontraron columnas numéricas")
else:
    print("⚠ Dataset no disponible")

## 2.5 Análisis de Distribuciones

Se analizan las distribuciones de las variables para identificar:
- Distribuciones normales
- Distribuciones sesgadas (positiva o negativa)
- Distribuciones multimodales
- Valores extremos

Este análisis es crucial para decidir qué técnica de normalización/estandarización aplicar (Criterio 9 de la Rúbrica).

In [ ]:
# Análisis de distribuciones
if 'df' in locals() and not df.empty:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        print("=" * 60)
        print("ANÁLISIS DE DISTRIBUCIONES")
        print("=" * 60)
        
        # Visualizar distribuciones con histogramas
        n_cols = min(3, len(numeric_cols))
        n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
        
        for idx, col in enumerate(numeric_cols[:n_rows*n_cols]):
            if idx < len(axes):
                df[col].hist(bins=30, ax=axes[idx], edgecolor='black')
                axes[idx].set_title(f'Distribución de {col}')
                axes[idx].set_xlabel(col)
                axes[idx].set_ylabel('Frecuencia')
                
                # Agregar línea de media
                mean_val = df[col].mean()
                axes[idx].axvline(mean_val, color='r', linestyle='--', label=f'Media: {mean_val:.2f}')
                axes[idx].legend()
        
        # Ocultar ejes vacíos
        for idx in range(len(numeric_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.savefig('../data/08_reporting/distribuciones_variables_numericas.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        # Análisis de sesgo (skewness)
        print("\n📊 Análisis de Sesgo (Skewness):")
        from scipy import stats
        skewness = df[numeric_cols].apply(lambda x: stats.skew(x.dropna()))
        
        skew_analysis = pd.DataFrame({
            'Sesgo': skewness,
            'Interpretación': skewness.apply(lambda s: 
                'Normal' if abs(s) < 0.5 else
                'Ligeramente sesgado' if abs(s) < 1 else
                'Moderadamente sesgado' if abs(s) < 2 else
                'Altamente sesgado'
            )
        })
        print(skew_analysis)
    else:
        print("⚠ No se encontraron columnas numéricas")
else:
    print("⚠ Dataset no disponible")

## 2.6 Análisis de Valores Faltantes

### Criterio 7 de la Rúbrica: Tratamiento de Missing Values

Se identifica y analiza la presencia de valores faltantes en el dataset para determinar la estrategia de tratamiento adecuada.

In [ ]:
# Análisis de valores faltantes
if 'df' in locals() and not df.empty:
    print("=" * 60)
    print("ANÁLISIS DE VALORES FALTANTES")
    print("=" * 60)
    
    # Usar función auxiliar
    missing_info = missing_summary(df)
    
    print("\n📊 Resumen de valores faltantes:")
    print(missing_info[missing_info['nulos'] > 0])
    
    # Visualización
    if missing_info['nulos'].sum() > 0:
        plt.figure(figsize=(12, 6))
        missing_cols = missing_info[missing_info['nulos'] > 0]
        
        plt.subplot(1, 2, 1)
        missing_cols['nulos'].plot(kind='barh')
        plt.title('Cantidad de valores faltantes por columna')
        plt.xlabel('Cantidad de nulos')
        
        plt.subplot(1, 2, 2)
        missing_cols['pct'].plot(kind='barh', color='orange')
        plt.title('Porcentaje de valores faltantes por columna')
        plt.xlabel('Porcentaje (%)')
        
        plt.tight_layout()
        plt.savefig('../data/08_reporting/analisis_missing_values.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\n💡 Estrategia de tratamiento:")
        print("  - Columnas con < 5% de missing: Imputación con mediana/moda")
        print("  - Columnas con 5-40% de missing: Análisis de patrón antes de imputar")
        print("  - Columnas con > 40% de missing: Considerar eliminación")
    else:
        print("\n✓ No se encontraron valores faltantes")
else:
    print("⚠ Dataset no disponible")

## 2.7 Análisis de Outliers (Valores Atípicos)

### Criterio 7 de la Rúbrica: Tratamiento de Outliers

Se detectan y analizan valores atípicos en las variables numéricas utilizando el método IQR (Rango Intercuartílico).

In [ ]:
# Análisis de outliers
if 'df' in locals() and not df.empty:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        print("=" * 60)
        print("ANÁLISIS DE OUTLIERS (VALORES ATÍPICOS)")
        print("=" * 60)
        
        outliers_summary = []
        
        for col in numeric_cols:
            outliers = detect_outliers_iqr(df, col)
            n_outliers = len(outliers)
            pct_outliers = (n_outliers / len(df)) * 100
            
            outliers_summary.append({
                'Variable': col,
                'Número de Outliers': n_outliers,
                'Porcentaje': f"{pct_outliers:.2f}%"
            })
        
        outliers_df = pd.DataFrame(outliers_summary)
        outliers_df = outliers_df[outliers_df['Número de Outliers'] > 0].sort_values('Número de Outliers', ascending=False)
        
        print("\n📊 Resumen de outliers detectados:")
        print(outliers_df)
        
        # Visualización con boxplots
        if len(outliers_df) > 0:
            n_cols = min(3, len(outliers_df))
            n_rows = (len(outliers_df) + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
            axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
            
            for idx, row in enumerate(outliers_df.head(n_rows*n_cols).itertuples()):
                if idx < len(axes):
                    sns.boxplot(y=df[row.Variable], ax=axes[idx])
                    axes[idx].set_title(f'Boxplot de {row.Variable}\n({row.Número_de_Outliers} outliers)')
            
            # Ocultar ejes vacíos
            for idx in range(len(outliers_df), len(axes)):
                if idx < len(axes):
                    axes[idx].set_visible(False)
            
            plt.tight_layout()
            plt.savefig('../data/08_reporting/analisis_outliers.png', dpi=150, bbox_inches='tight')
            plt.show()
            
            print("\n💡 Estrategia de tratamiento:")
            print("  - Outliers < 1%: Mantener (pueden ser valores legítimos)")
            print("  - Outliers 1-5%: Analizar contexto antes de decidir")
            print("  - Outliers > 5%: Considerar capping o eliminación")
    else:
        print("⚠ No se encontraron columnas numéricas")
else:
    print("⚠ Dataset no disponible")

## 2.8 Análisis de Variables Categóricas

Se analizan las variables categóricas para identificar:
- Distribución de categorías
- Categorías raras o inconsistentes
- Errores de captura

In [ ]:
# Análisis de variables categóricas
if 'df' in locals() and not df.empty:
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if categorical_cols:
        print("=" * 60)
        print("ANÁLISIS DE VARIABLES CATEGÓRICAS")
        print("=" * 60)
        
        for col in categorical_cols[:10]:  # Limitar a 10 para no saturar
            print(f"\n📊 {col}:")
            value_counts = df[col].value_counts()
            print(value_counts.head(10))
            
            # Identificar categorías raras (< 1% de los datos)
            rare_categories = value_counts[value_counts < len(df) * 0.01]
            if len(rare_categories) > 0:
                print(f"  ⚠ Categorías raras (< 1%): {len(rare_categories)} categorías")
        
        # Visualización
        n_cols = min(2, len(categorical_cols))
        n_rows = (len(categorical_cols) + n_cols - 1) // n_cols
        
        if n_rows > 0:
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
            axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
            
            for idx, col in enumerate(categorical_cols[:n_rows*n_cols]):
                if idx < len(axes):
                    value_counts = df[col].value_counts().head(10)
                    value_counts.plot(kind='bar', ax=axes[idx])
                    axes[idx].set_title(f'Distribución de {col}')
                    axes[idx].set_xlabel(col)
                    axes[idx].set_ylabel('Frecuencia')
                    axes[idx].tick_params(axis='x', rotation=45)
            
            # Ocultar ejes vacíos
            for idx in range(len(categorical_cols), len(axes)):
                if idx < len(axes):
                    axes[idx].set_visible(False)
            
            plt.tight_layout()
            plt.savefig('../data/08_reporting/distribuciones_categoricas.png', dpi=150, bbox_inches='tight')
            plt.show()
    else:
        print("⚠ No se encontraron variables categóricas")
else:
    print("⚠ Dataset no disponible")

## 2.9 Análisis de Correlaciones

Se analiza la correlación entre variables numéricas para identificar relaciones y posibles multicolinealidad.

In [ ]:
# Análisis de correlación
if 'df' in locals() and not df.empty:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(numeric_cols) > 1:
        print("=" * 60)
        print("ANÁLISIS DE CORRELACIÓN")
        print("=" * 60)
        
        # Matriz de correlación
        corr_matrix = df[numeric_cols].corr()
        
        # Visualización con mapa de calor
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                   center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('Matriz de Correlación entre Variables Numéricas')
        plt.tight_layout()
        plt.savefig('../data/08_reporting/matriz_correlacion.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        # Identificar correlaciones fuertes
        print("\n📊 Correlaciones fuertes (|r| > 0.7):")
        strong_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = corr_matrix.iloc[i, j]
                if abs(corr_val) > 0.7:
                    strong_corr.append({
                        'Variable 1': corr_matrix.columns[i],
                        'Variable 2': corr_matrix.columns[j],
                        'Correlación': f"{corr_val:.3f}"
                    })
        
        if strong_corr:
            print(pd.DataFrame(strong_corr))
        else:
            print("  No se encontraron correlaciones fuertes")
    else:
        print("⚠ Se necesitan al menos 2 variables numéricas para análisis de correlación")
else:
    print("⚠ Dataset no disponible")

---

# FASE 3: DATA PREPARATION
## Preparación de los Datos

### Criterio 5 de la Rúbrica: Limpieza y Preparación según Buenas Prácticas

En esta fase se aplican las transformaciones necesarias para preparar los datos para el modelado, siguiendo las mejores prácticas de la industria.

## 3.1 Tratamiento de Valores Faltantes

### Criterio 7 de la Rúbrica: Tratamiento de Missing Values según Naturaleza de los Datos

**Justificación de la estrategia:**
- Variables numéricas: Se imputan con la mediana (robusta a outliers) o media según distribución
- Variables categóricas: Se imputan con la moda o se crea categoría "Unknown"
- Variables con > 40% de missing: Se evalúa eliminación o tratamiento especial

In [ ]:
# Tratamiento de valores faltantes
if 'df' in locals() and not df.empty:
    print("=" * 60)
    print("TRATAMIENTO DE VALORES FALTANTES")
    print("=" * 60)
    
    # Crear copia para preprocesamiento
    df_clean = df.copy()
    
    # Identificar columnas numéricas y categóricas
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Obtener información de missing antes del tratamiento
    missing_before = missing_summary(df_clean)
    
    print("\n📊 Valores faltantes ANTES del tratamiento:")
    print(missing_before[missing_before['nulos'] > 0])
    
    # Estrategia: Imputar numéricas con mediana
    numeric_missing = [col for col in numeric_cols if df_clean[col].isnull().sum() > 0]
    if numeric_missing:
        print(f"\n🔧 Imputando {len(numeric_missing)} variables numéricas con mediana...")
        df_clean = impute_numeric_median(df_clean, numeric_missing)
    
    # Estrategia: Imputar categóricas con moda
    categorical_missing = [col for col in categorical_cols if df_clean[col].isnull().sum() > 0]
    if categorical_missing:
        print(f"\n🔧 Imputando {len(categorical_missing)} variables categóricas con moda...")
        df_clean = impute_categorical_mode(df_clean, categorical_missing)
    
    # Verificar después del tratamiento
    missing_after = missing_summary(df_clean)
    
    print("\n✓ Valores faltantes DESPUÉS del tratamiento:")
    remaining_missing = missing_after[missing_after['nulos'] > 0]
    if len(remaining_missing) > 0:
        print(remaining_missing)
    else:
        print("  No quedan valores faltantes")
    
    print("\n💾 Guardando datos limpios...")
    save_csv(df_clean, '../data/02_intermediate/df_cleaned.csv')
    print("✓ Datos guardados en data/02_intermediate/df_cleaned.csv")
else:
    print("⚠ Dataset no disponible")

## 3.2 Tratamiento de Outliers

### Criterio 7 de la Rúbrica: Tratamiento de Outliers según Naturaleza de los Datos

**Justificación de la estrategia:**
- Se utiliza el método IQR (Rango Intercuartílico) para detectar outliers
- Se aplica "capping" (limitar valores extremos) en lugar de eliminación para conservar información
- Esta estrategia preserva el tamaño del dataset mientras reduce el impacto de valores extremos

In [ ]:
# Tratamiento de outliers
if 'df_clean' in locals() and not df_clean.empty:
    print("=" * 60)
    print("TRATAMIENTO DE OUTLIERS")
    print("=" * 60)
    
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    
    # Identificar columnas con outliers significativos (> 1%)
    cols_with_outliers = []
    for col in numeric_cols:
        outliers = detect_outliers_iqr(df_clean, col)
        pct = (len(outliers) / len(df_clean)) * 100
        if pct > 1:  # Más del 1% de outliers
            cols_with_outliers.append(col)
    
    if cols_with_outliers:
        print(f"\n📊 Columnas con outliers significativos: {len(cols_with_outliers)}")
        print(f"  Columnas: {', '.join(cols_with_outliers)}")
        
        # Aplicar capping (limitar valores extremos)
        print("\n🔧 Aplicando capping a outliers (método IQR)...")
        for col in cols_with_outliers:
            before_stats = df_clean[col].describe()
            df_clean = cap_outliers(df_clean, col, k=1.5)
            after_stats = df_clean[col].describe()
            
            print(f"\n  {col}:")
            print(f"    Min antes: {before_stats['min']:.2f} → Min después: {after_stats['min']:.2f}")
            print(f"    Max antes: {before_stats['max']:.2f} → Max después: {after_stats['max']:.2f}")
        
        print("\n✓ Outliers tratados con capping")
    else:
        print("\n✓ No se encontraron columnas con outliers significativos")
    
    # Guardar datos con outliers tratados
    save_csv(df_clean, '../data/02_intermediate/df_cleaned_no_outliers.csv')
    print("\n💾 Datos guardados en data/02_intermediate/df_cleaned_no_outliers.csv")
else:
    print("⚠ Dataset limpio no disponible")

## 3.3 Limpieza de Variables Categóricas

Se limpian y normalizan las variables categóricas:
- Corrección de errores de captura
- Agrupación de categorías raras
- Normalización de formatos

In [ ]:
# Limpieza de variables categóricas
if 'df_clean' in locals() and not df_clean.empty:
    print("=" * 60)
    print("LIMPIEZA DE VARIABLES CATEGÓRICAS")
    print("=" * 60)
    
    categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if categorical_cols:
        print(f"\n📊 Variables categóricas a limpiar: {len(categorical_cols)}")
        
        for col in categorical_cols:
            # Identificar categorías raras (< 1% de frecuencia)
            value_counts = df_clean[col].value_counts()
            rare_threshold = len(df_clean) * 0.01
            rare_categories = value_counts[value_counts < rare_threshold].index.tolist()
            
            if len(rare_categories) > 0 and len(rare_categories) < len(value_counts) * 0.5:
                # Agrupar categorías raras en "Other"
                print(f"\n  {col}: Agrupando {len(rare_categories)} categorías raras en 'Other'")
                df_clean[col] = df_clean[col].replace(rare_categories, 'Other')
            
            # Normalizar espacios y mayúsculas/minúsculas
            if df_clean[col].dtype == 'object':
                df_clean[col] = df_clean[col].str.strip().str.title()
        
        print("\n✓ Variables categóricas limpiadas")
    else:
        print("\n⚠ No se encontraron variables categóricas")
    
    # Guardar datos con categóricas limpiadas
    save_csv(df_clean, '../data/02_intermediate/df_cleaned_categorical.csv')
    print("\n💾 Datos guardados en data/02_intermediate/df_cleaned_categorical.csv")
else:
    print("⚠ Dataset limpio no disponible")

## 3.4 Creación de Variables Derivadas (Feature Engineering)

Se crean nuevas variables que pueden ser útiles para el modelado:
- Variables de tiempo (día de semana, mes, año)
- Variables de agregación
- Variables de interacción

In [ ]:
# Creación de variables derivadas
if 'df_clean' in locals() and not df_clean.empty:
    print("=" * 60)
    print("CREACIÓN DE VARIABLES DERIVADAS")
    print("=" * 60)
    
    # Usar función auxiliar para crear features de e-commerce
    df_features = create_ecommerce_features(df_clean.copy())
    
    print("\n✓ Variables derivadas creadas")
    print(f"  Dimensiones: {df_features.shape[0]} filas × {df_features.shape[1]} columnas")
    
    # Mostrar nuevas columnas creadas
    new_cols = [col for col in df_features.columns if col not in df_clean.columns]
    if new_cols:
        print(f"\n📊 Nuevas variables creadas ({len(new_cols)}):")
        for col in new_cols:
            print(f"  - {col}")
    
    df_clean = df_features
    
    # Guardar datos con features
    save_csv(df_clean, '../data/02_intermediate/df_with_features.csv')
    print("\n💾 Datos guardados en data/02_intermediate/df_with_features.csv")
else:
    print("⚠ Dataset limpio no disponible")

## 3.5 Encoding de Variables Categóricas

Los modelos de machine learning requieren variables numéricas. Se aplica encoding a las variables categóricas:
- **One-hot encoding**: Para variables categóricas nominales (sin orden)
- **Label encoding**: Para variables categóricas ordinales (con orden)

In [ ]:
# Encoding de variables categóricas
if 'df_clean' in locals() and not df_clean.empty:
    print("=" * 60)
    print("ENCODING DE VARIABLES CATEGÓRICAS")
    print("=" * 60)
    
    categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Excluir columnas que no deben ser codificadas (IDs, etc.)
    exclude_cols = [col for col in categorical_cols if 'id' in col.lower() or 'code' in col.lower()]
    categorical_cols = [col for col in categorical_cols if col not in exclude_cols]
    
    if categorical_cols:
        print(f"\n📊 Variables categóricas a codificar: {len(categorical_cols)}")
        print(f"  Columnas: {', '.join(categorical_cols[:5])}{'...' if len(categorical_cols) > 5 else ''}")
        
        # Aplicar one-hot encoding
        print("\n🔧 Aplicando one-hot encoding...")
        df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, prefix_sep='_')
        
        print(f"\n✓ Encoding completado")
        print(f"  Columnas antes: {df_clean.shape[1]}")
        print(f"  Columnas después: {df_encoded.shape[1]}")
        print(f"  Nuevas columnas: {df_encoded.shape[1] - df_clean.shape[1]}")
        
        # Mostrar ejemplo
        print("\n👀 Ejemplo de columnas codificadas:")
        encoded_cols = [col for col in df_encoded.columns if any(cat_col in col for cat_col in categorical_cols[:3])]
        if encoded_cols:
            print(df_encoded[encoded_cols[:5]].head())
    else:
        print("\n⚠ No se encontraron variables categóricas para codificar")
        df_encoded = df_clean.copy()
    
    # Guardar datos codificados
    save_csv(df_encoded, '../data/02_intermediate/df_encoded.csv')
    print("\n💾 Datos guardados en data/02_intermediate/df_encoded.csv")
else:
    print("⚠ Dataset limpio no disponible")

## 3.6 Normalización/Estandarización

### Criterio 9 de la Rúbrica: Normalización/Estandarización según Distribución

**Justificación de la técnica elegida:**

1. **StandardScaler (Estandarización)**: 
   - Se usa cuando las variables tienen distribución aproximadamente normal
   - Transforma a media=0 y desviación estándar=1
   - Apropiado para algoritmos que asumen normalidad (regresión lineal, SVM)

2. **MinMaxScaler (Normalización)**:
   - Se usa cuando las variables tienen distribución sesgada o no normal
   - Escala a rango [0, 1]
   - Apropiado para algoritmos sensibles a la escala (redes neuronales, k-means)

**Decisión:** Basado en el análisis de distribuciones realizado en la Fase 2, se aplicará la técnica más adecuada para cada variable.

In [ ]:
# Normalización/Estandarización
if 'df_encoded' in locals() and not df_encoded.empty:
    print("=" * 60)
    print("NORMALIZACIÓN/ESTANDARIZACIÓN")
    print("=" * 60)
    
    # Seleccionar solo columnas numéricas (excluir las codificadas con one-hot)
    numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
    
    # Excluir columnas binarias (0/1) de one-hot encoding
    binary_cols = [col for col in numeric_cols if df_encoded[col].nunique() == 2 and set(df_encoded[col].unique()).issubset({0, 1})]
    numeric_cols = [col for col in numeric_cols if col not in binary_cols]
    
    # Excluir IDs y columnas que no deben escalarse
    exclude_cols = [col for col in numeric_cols if 'id' in col.lower()]
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    if numeric_cols:
        print(f"\n📊 Variables numéricas a escalar: {len(numeric_cols)}")
        
        # Analizar distribuciones para decidir técnica
        from scipy import stats
        
        # Variables para StandardScaler (distribución normal)
        cols_standard = []
        # Variables para MinMaxScaler (distribución sesgada)
        cols_minmax = []
        
        for col in numeric_cols:
            skewness = stats.skew(df_encoded[col].dropna())
            # Si el sesgo es bajo (< 1), usar StandardScaler
            if abs(skewness) < 1:
                cols_standard.append(col)
            else:
                cols_minmax.append(col)
        
        print(f"\n📈 Estrategia de escalado:")
        print(f"  - StandardScaler (distribución normal): {len(cols_standard)} variables")
        print(f"  - MinMaxScaler (distribución sesgada): {len(cols_minmax)} variables")
        
        # Aplicar StandardScaler
        if cols_standard:
            print(f"\n🔧 Aplicando StandardScaler a {len(cols_standard)} variables...")
            scaler_standard = StandardScaler()
            df_encoded[cols_standard] = scaler_standard.fit_transform(df_encoded[cols_standard])
        
        # Aplicar MinMaxScaler
        if cols_minmax:
            print(f"\n🔧 Aplicando MinMaxScaler a {len(cols_minmax)} variables...")
            scaler_minmax = MinMaxScaler()
            df_encoded[cols_minmax] = scaler_minmax.fit_transform(df_encoded[cols_minmax])
        
        # Verificar resultado
        print("\n✓ Escalado completado")
        print("\n📊 Estadísticos después del escalado (primeras 5 variables):")
        print(df_encoded[numeric_cols[:5]].describe())
        
        # Guardar scalers para uso futuro
        import joblib
        if cols_standard:
            joblib.dump(scaler_standard, '../data/06_models/scaler_standard.pkl')
        if cols_minmax:
            joblib.dump(scaler_minmax, '../data/06_models/scaler_minmax.pkl')
        print("\n💾 Scalers guardados en data/06_models/")
    else:
        print("\n⚠ No se encontraron variables numéricas para escalar")
    
    # Guardar datos finales procesados
    save_csv(df_encoded, '../data/03_processed/df_processed_final.csv')
    print("\n💾 Datos finales procesados guardados en data/03_processed/df_processed_final.csv")
    
    print("\n" + "=" * 60)
    print("✅ PREPROCESAMIENTO COMPLETADO")
    print("=" * 60)
    print(f"\n📊 Resumen final:")
    print(f"  - Filas: {df_encoded.shape[0]:,}")
    print(f"  - Columnas: {df_encoded.shape[1]}")
    print(f"  - Valores faltantes: {df_encoded.isnull().sum().sum()}")
    print(f"  - Duplicados: {df_encoded.duplicated().sum()}")
else:
    print("⚠ Dataset codificado no disponible")

---

# Comparación Antes/Después del Preprocesamiento

### Criterio 6 de la Rúbrica: Documentación Comparando Resultados

Se compara el estado de los datos antes y después del preprocesamiento para documentar el impacto de las transformaciones.

In [ ]:
# Comparación antes/después
if 'df' in locals() and 'df_encoded' in locals() and not df.empty and not df_encoded.empty:
    print("=" * 60)
    print("COMPARACIÓN ANTES/DESPUÉS DEL PREPROCESAMIENTO")
    print("=" * 60)
    
    comparison = pd.DataFrame({
        'Métrica': [
            'Número de filas',
            'Número de columnas',
            'Valores faltantes',
            'Duplicados',
            'Variables numéricas',
            'Variables categóricas'
        ],
        'Antes': [
            df.shape[0],
            df.shape[1],
            df.isnull().sum().sum(),
            df.duplicated().sum(),
            len(df.select_dtypes(include=[np.number]).columns),
            len(df.select_dtypes(include=['object', 'category']).columns)
        ],
        'Después': [
            df_encoded.shape[0],
            df_encoded.shape[1],
            df_encoded.isnull().sum().sum(),
            df_encoded.duplicated().sum(),
            len(df_encoded.select_dtypes(include=[np.number]).columns),
            len(df_encoded.select_dtypes(include=['object', 'category']).columns)
        ]
    })
    
    comparison['Cambio'] = comparison['Después'] - comparison['Antes']
    
    print("\n📊 Tabla comparativa:")
    print(comparison.to_string(index=False))
    
    print("\n📈 Impacto de las transformaciones:")
    print(f"  ✓ Valores faltantes eliminados: {comparison.loc[2, 'Antes'] - comparison.loc[2, 'Después']}")
    print(f"  ✓ Variables categóricas codificadas: {comparison.loc[5, 'Antes']} → {comparison.loc[5, 'Después']}")
    print(f"  ✓ Variables escaladas: Todas las numéricas fueron normalizadas/estandarizadas")
    
    # Guardar comparación
    comparison.to_csv('../data/08_reporting/comparacion_preprocesamiento.csv', index=False)
    print("\n💾 Comparación guardada en data/08_reporting/comparacion_preprocesamiento.csv")
else:
    print("⚠ Datasets no disponibles para comparación")

---

# Conclusiones y Próximos Pasos

## Resumen del Proceso

Este notebook ha cubierto las **primeras 3 fases de CRISP-DM**:

1. ✅ **Business Understanding**: Se definió el contexto del negocio y los objetivos
2. ✅ **Data Understanding**: Se realizó un análisis exploratorio completo
3. ✅ **Data Preparation**: Se aplicaron transformaciones y preprocesamiento

## Criterios de la Rúbrica Cubiertos

- ✅ **Criterio 1**: Uso de CRISP-DM en Jupyter Notebook
- ✅ **Criterio 2**: Identificación de target para regresión
- ✅ **Criterio 3**: Identificación de target para clasificación
- ✅ **Criterio 4**: Uso de librerías Python (numpy, scikit-learn, matplotlib, seaborn)
- ✅ **Criterio 5**: Limpieza y preparación según buenas prácticas
- ✅ **Criterio 6**: Documentación comparando resultados
- ✅ **Criterio 7**: Tratamiento de outliers y missing values
- ✅ **Criterio 8**: Estadísticos de tendencia central y dispersión
- ✅ **Criterio 9**: Normalización/estandarización según distribución
- ✅ **Criterio 10**: Documentación con Markdown justificando técnicas

## Próximos Pasos

1. **Fase 4 - Modeling**: Entrenar modelos de regresión y clasificación
2. **Fase 5 - Evaluation**: Evaluar el desempeño de los modelos
3. **Fase 6 - Deployment**: Documentación final y deployment

---

**Nota:** Este notebook está preparado para trabajar con los datasets de brazilian-ecommerce. Una vez que se carguen los datos reales, se ajustarán los nombres de columnas y las transformaciones específicas según la estructura real de los datos.